# Trade-signal outcome tracker

Separate from the dashboard on purpose. The dashboard says *what looks like a trade now*;
this says *did the trades it flagged actually work*. It does not import, modify or depend on
the dashboard's UI — it re-runs the dashboard's own signal engines on truncated data.

**How it works**

1. Rewind the price/IV history to a past date `AS_OF` and re-run `build_signals`. Every engine
   reads the last row of each series, so a truncated frame reproduces exactly what the dashboard
   printed that day — confidence hit-rates included, with no look-ahead.
2. Take the **5 highest-confidence** signals, then top up so **every engine that fired gets at
   least one trade** (labelled `ENGINE`) and at least **3 trades touch metals or energy**
   (labelled `SECTOR`) — even when nothing in those buckets scored well. Without the engine
   pass, one engine's good week crowds the others out of the results entirely.
3. Mark each trade forward over the next `HORIZON` sessions, entry close to exit close, using the
   payoff that matches what the engine was betting on (see the notes at the bottom).

**To run in BQuant:** upload this notebook next to `commodities_vol_rv_dashboard.ipynb` and run
every cell top to bottom. There is nothing else to install — no `.py` files, no imports beyond
what the dashboard already uses. Nothing here writes to the dashboard.

If you would rather not keep two notebooks, paste cells 1–7 of this one onto the end of the
dashboard notebook instead; the setup cell detects that the engines are already defined and
skips loading them from disk.

In [ ]:
# =====================================================================
# TRACKER LIBRARY — generated, do not edit here
# =====================================================================
# Source of truth is dev/tracker_source.py; this cell is produced by
# dev/build_notebook.py. It lives inline because BQuant cannot import a
# .py file — this notebook plus the dashboard notebook is all you need.
# Run it once, then carry on to the next cell.

"""
Trade-signal outcome tracker
============================

Deliberately separate from the dashboard. The dashboard answers "what looks
like a trade right now"; this answers "did the trades it flagged actually
work". Nothing in here changes the dashboard — it *reuses* its engines by
exec'ing the notebook's non-UI code cells, so the signals being scored are
byte-identical to the ones the dashboard prints.

How the replay works
--------------------
Every engine in `build_signals` reads the LAST row of each series. So handing
it price/IV frames truncated at date T reproduces exactly what the dashboard
would have shown on T — including the confidence hit-rates, which are computed
from `shift(-h)` forward windows and therefore only ever see data inside the
truncated frame. There is no look-ahead in the replay.

    signals_asof(T)  ->  pick trades  ->  score them over the next N sessions

Which trades get tracked
-----------------------
The top TOP_N signals by confidence, then two top-up passes: one trade from
every engine that fired, and enough metals/energy trades to reach SECTOR_N.
The top-ups are what make the by-engine and by-sector breakdowns readable —
without them a single engine's good week fills all five slots and the others
never get a recorded outcome. They are labelled in the report so they can be
separated from the picks that got in on merit.

Outcome definitions, per engine
-------------------------------
Each engine bets on a different thing, so each gets the P&L that matches the
bet. All are marked from the entry close to the exit close, no costs, no
sizing — this measures signal direction, not a tradable P&L.

    IV mean-reversion   BUY VOL wins if IV rose;  SELL VOL wins if IV fell.
                        P&L = ±(IV_exit - IV_entry), in vol points.

    Variance risk prem. The real test: did realized vol over the holding week
                        come in under the implied vol quoted at entry?
                        P&L = ±(IV_entry - RV_realized_fwd), in vol points.
                        (SELL VOL is long that spread, BUY VOL is short it.)

    Vol dispersion      Short the rich leg's vol, long the cheap leg's.
                        P&L = (ΔIV_cheap - ΔIV_rich), in vol points — positive
                        when the spread converged.

    Correlation RV      Long the laggard, short the outperformer, equal notional.
                        P&L = ret(long) - ret(short), in %. Positive = gap closed.

    Lead-lag catch-up   Directional in the follower.
                        P&L = ±ret(follower), in %.

A "week" is HORIZON_DAYS trading sessions, not calendar days, so holidays
don't silently shorten the holding period.
"""

import json
import os
import re

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Defaults
# ---------------------------------------------------------------------------
HORIZON_DAYS = 5                                     # one trading week
TOP_N = 5                                            # highest-confidence picks
SECTORS = ("Energy", "Precious Metals", "Base Metals")
SECTOR_N = 3                                         # metals/energy floor
ENGINE_N = 1                                         # trades tracked per engine, minimum

VOL_ENGINES = ("IV mean-reversion", "Variance risk premium", "Vol dispersion (pairs)")

# Same palette as the dashboard so the report drops into the same environment
# without looking like a different tool. Overridden by the host namespace when
# one is supplied (see Tracker.__init__).
THEME = dict(BG="#0B0E14", PANEL="#151B26", GRID="#2C3644", TXT="#F2F6FC",
             MUTED="#95A3B8", GREEN="#25D07A", RED="#FF5B5B", AMBER="#FFC44D",
             BLUE="#4DB6FF", PURPLE="#C58CFF", TEAL="#2FD9C6")

ENGINE_COLOR = {"IV mean-reversion": "#4DB6FF", "Variance risk premium": "#FFC44D",
                "Vol dispersion (pairs)": "#2FD9C6", "Correlation RV": "#C58CFF",
                "Lead-lag catch-up": "#25D07A"}

# The dashboard's five engines, in its own display order.
ENGINES = tuple(ENGINE_COLOR)

# Markers identifying the dashboard cells worth importing. The RENDERERS and
# CONTROLS cells are skipped on purpose — they build widgets and fire a
# Bloomberg pull on import.
DASHBOARD_SECTIONS = ("CONFIG —", "ANALYTICS —", "DATA LAYER —")

# A cell containing any of these builds or wires the UI: never exec it, whatever
# its header says. `on_click` in particular is what triggers the Bloomberg pull.
UI_MARKERS = ("widgets.Tab(", ".on_click(", ".observe(")

# ...and these identify the cells we do want even if their headers were renamed.
ENGINE_MARKERS = ("def build_signals", "bq = bql.Service()", "def fetch_all")


def _wanted_cell(src, sections):
    if any(u in src for u in UI_MARKERS):
        return False
    return any(s in src[:400] for s in sections) or any(k in src for k in ENGINE_MARKERS)


def load_dashboard(path, ns=None, sections=DASHBOARD_SECTIONS):
    """Exec the dashboard notebook's non-UI code cells into a namespace dict."""
    with open(path) as fh:
        nb = json.load(fh)
    ns = {} if ns is None else ns
    loaded = []
    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell["source"])
        if not _wanted_cell(src, sections):
            continue
        exec(compile(src, "%s#cell%d" % (os.path.basename(path), i), "exec"), ns)
        loaded.append(i)
    if "build_signals" not in ns:
        raise RuntimeError(
            "%s has no cell defining build_signals — is that the dashboard notebook?" % path)
    ns.setdefault("__loaded_cells__", loaded)
    return ns


def _defines_engines(path):
    try:
        with open(path) as fh:
            return "def build_signals" in fh.read()
    except (OSError, UnicodeDecodeError, ValueError):
        return False


def find_dashboard(preferred="commodities_vol_rv_dashboard.ipynb", extra_dirs=()):
    """Locate the dashboard notebook on disk.

    BQuant does not guarantee which directory a notebook's kernel starts in, so
    rather than assuming a path this looks for any .ipynb that actually defines
    build_signals — which survives the file being renamed or moved a folder up.
    Searched: the working directory, its parents, the home directory, and one
    level of subfolders under each.
    """
    cwd = os.getcwd()
    roots, p = [cwd] + list(extra_dirs) + [os.path.expanduser("~")], cwd
    for _ in range(3):
        p = os.path.dirname(p) or os.sep
        roots.append(p)

    searched, hits = [], []
    for root in roots:
        if not root or not os.path.isdir(root) or root in searched:
            continue
        searched.append(root)
        try:
            entries = sorted(os.listdir(root))
        except OSError:
            continue
        dirs = [os.path.join(root, e) for e in entries
                if not e.startswith(".") and os.path.isdir(os.path.join(root, e))]
        for folder in [root] + dirs[:40]:
            try:
                names = sorted(os.listdir(folder))
            except OSError:
                continue
            for fn in names:
                if fn.endswith(".ipynb") and _defines_engines(os.path.join(folder, fn)):
                    hits.append(os.path.join(folder, fn))

    if not hits:
        raise FileNotFoundError(
            "Could not find the dashboard notebook (no .ipynb defining build_signals).\n"
            "Searched: %s\n"
            "Fix: put the dashboard notebook in the same folder as this one, or load it "
            "explicitly with  NS = load_dashboard('/full/path/to/dashboard.ipynb')"
            % ", ".join(searched))
    for h in hits:
        if os.path.basename(h) == preferred:
            return h
    return hits[0]


# ---------------------------------------------------------------------------
# Tracker
# ---------------------------------------------------------------------------
class Tracker:
    """Replays the dashboard's signal engines as of a past date and scores them.

    `ns` is the namespace produced by load_dashboard() (or the dashboard
    notebook's own globals() if you are running inside it).
    """

    def __init__(self, ns, horizon=HORIZON_DAYS):
        missing = [k for k in ("build_signals", "realized_vol", "NAME", "ASSET_CLASS")
                   if k not in ns]
        if missing:
            raise RuntimeError("namespace is missing %s — load the dashboard's "
                               "CONFIG and ANALYTICS cells first" % ", ".join(missing))
        self.ns = ns
        self.horizon = int(horizon)
        self.NAME = ns["NAME"]
        self.ASSET_CLASS = ns["ASSET_CLASS"]
        self.build_signals = ns["build_signals"]
        self.realized_vol = ns["realized_vol"]
        self.by_name = {v: k for k, v in self.NAME.items()}
        self.theme = {k: ns.get(k, v) for k, v in THEME.items()}

    # -- dates -------------------------------------------------------------
    def _pos(self, index, when):
        """Index position of the last session on or before `when` (-1 if none)."""
        return int(index.searchsorted(pd.Timestamp(when), side="right")) - 1

    def entry_exit(self, close, asof, horizon=None):
        """(entry_date, exit_date, sessions_held). exit is None if unseasoned."""
        h = self.horizon if horizon is None else int(horizon)
        idx = close.index
        i = self._pos(idx, asof)
        if i < 0:
            raise ValueError("no price history on or before %s" % asof)
        j = min(i + h, len(idx) - 1)
        held = j - i
        return idx[i], (idx[j] if held > 0 else None), held

    # -- replay ------------------------------------------------------------
    def signals_asof(self, px, iv, cfg, asof):
        """Every signal the dashboard would have printed on `asof`, one per row."""
        px_a = {k: v.loc[:pd.Timestamp(asof)] for k, v in px.items()}
        iv_a = iv.loc[:pd.Timestamp(asof)]
        res = self.build_signals(px_a, iv_a, cfg)
        frames = []
        for engine, df in res["engines"].items():
            if df is None or df.empty:
                continue
            d = df.copy()
            d["engine"] = engine
            frames.append(d)
        if not frames:
            return pd.DataFrame()
        out = pd.concat(frames, ignore_index=True, sort=False)
        return out.sort_values(["conf", "score"], ascending=[False, False]).reset_index(drop=True)

    # -- leg resolution ----------------------------------------------------
    def _tk(self, nm):
        return self.by_name.get((nm or "").strip())

    def legs(self, engine, name, side):
        """-> (kind, [(ticker, signed weight), ...]).

        Signs are position signs: +1 long (long vol / long the asset), -1 short.
        The engines emit their legs only as display text, so this parses the
        `side` string they build; an unparseable row is returned as (None, [])
        and scored as N.A. rather than guessed at.
        """
        side = (side or "").strip()
        if engine in ("IV mean-reversion", "Variance risk premium"):
            tk = self._tk(name)
            sgn = 1.0 if side.upper().startswith("BUY") else -1.0
            return ("vol_single", [(tk, sgn)]) if tk else (None, [])
        if engine == "Vol dispersion (pairs)":
            m = re.match(r"^SELL (.+?) VOL / BUY (.+?) VOL$", side)
            if not m:
                return (None, [])
            rich, cheap = self._tk(m.group(1)), self._tk(m.group(2))
            return ("vol_pair", [(rich, -1.0), (cheap, 1.0)]) if rich and cheap else (None, [])
        if engine == "Correlation RV":
            m = re.match(r"^BUY (.+?) / SELL (.+?)$", side)
            if not m:
                return (None, [])
            lng, sht = self._tk(m.group(1)), self._tk(m.group(2))
            return ("px_pair", [(lng, 1.0), (sht, -1.0)]) if lng and sht else (None, [])
        if engine == "Lead-lag catch-up":
            m = re.match(r"^(BUY|SELL) (.+?)$", side)
            if not m:
                return (None, [])
            tk = self._tk(m.group(2))
            sgn = 1.0 if m.group(1) == "BUY" else -1.0
            return ("px_single", [(tk, sgn)]) if tk else (None, [])
        return (None, [])

    def classes(self, legs):
        return [self.ASSET_CLASS.get(tk, "Other") for tk, _ in legs]

    # -- selection ---------------------------------------------------------
    def select(self, sig, top_n=TOP_N, sectors=SECTORS, sector_n=SECTOR_N,
               engine_n=ENGINE_N, engines=None):
        """Pick the trades to track, in three passes:

        1. the `top_n` highest-confidence signals on the board;
        2. top up to `engine_n` trades from every engine that fired, so each
           engine has an outcome to be judged on rather than being crowded out
           by whichever one happens to score highest that week;
        3. top up so at least `sector_n` trades touch metals/energy.

        Deduped on the set of underlyings: the same asset flagged by two engines
        is one bet, not two, so only its higher-confidence version is tracked.
        Both top-ups take the best available within their bucket and are included
        even when confidence is poor — that is the point. A coin-flip signal the
        desk would still look at deserves a recorded outcome, and an engine only
        looks good if you also count the weeks its best idea was mediocre.
        """
        if sig is None or sig.empty:
            return pd.DataFrame()
        engines = tuple(engines) if engines is not None else ENGINES
        rows, seen = [], set()

        def add(r, basis):
            kind, legs = self.legs(r["engine"], r["name"], r["side"])
            key = frozenset(tk for tk, _ in legs) if legs else ("?", r["engine"], r["name"])
            if key in seen:
                return False
            seen.add(key)
            d = dict(r)
            d.update(basis=basis, kind=kind, legs=legs, sectors=self.classes(legs))
            rows.append(d)
            return True

        for _, r in sig.iterrows():
            if len(rows) >= top_n:
                break
            add(r, "top confidence")

        for eng in engines:
            pool = sig[sig["engine"] == eng]
            for _, r in pool.iterrows():
                if sum(d["engine"] == eng for d in rows) >= engine_n:
                    break
                add(r, "engine coverage")

        def in_sector(d):
            return any(c in sectors for c in d["sectors"])

        for _, r in sig.iterrows():
            if sum(in_sector(d) for d in rows) >= sector_n:
                break
            kind, legs = self.legs(r["engine"], r["name"], r["side"])
            if not legs or not any(self.ASSET_CLASS.get(tk) in sectors for tk, _ in legs):
                continue
            add(r, "metals/energy quota")

        out = pd.DataFrame(rows)
        return out.reset_index(drop=True)

    # -- scoring -----------------------------------------------------------
    def score(self, trades, px, iv, cfg, asof, horizon=None):
        """Mark every selected trade from entry close to exit close."""
        if trades is None or trades.empty:
            return pd.DataFrame()
        h = self.horizon if horizon is None else int(horizon)
        close = px["close"]
        entry, exit_, held = self.entry_exit(close, asof, h)
        ivf = iv.reindex(close.index).ffill()

        # Realized vol measured over a window exactly as long as the holding
        # period: its value AT the exit date is the vol actually delivered
        # between entry and exit — the number a vol seller is marked against.
        rv_fwd = self.realized_vol(px, cfg["rv_estimator"], max(2, held)) if held else None

        def lvl(frame, tk, when):
            try:
                v = frame.at[when, tk]
            except (KeyError, IndexError):
                return np.nan
            return float(v) if pd.notna(v) else np.nan

        out = []
        for _, t in trades.iterrows():
            d = dict(t)
            d.update(asof=entry, entry_date=entry, exit_date=exit_, sessions=held,
                     pnl=np.nan, unit="", outcome="N.A.", detail="", note="")
            legs, kind = t["legs"], t["kind"]

            if not legs or kind is None:
                d["note"] = "could not resolve the trade's legs from the signal text"
                out.append(d); continue
            if exit_ is None:
                d["outcome"] = "OPEN"
                d["note"] = "no sessions after %s yet" % pd.Timestamp(entry).date()
                out.append(d); continue

            if kind in ("vol_single", "vol_pair"):
                d["unit"] = "vol pts"
                parts, pnl, bad = [], 0.0, False
                for tk, sgn in legs:
                    iv0, iv1 = lvl(ivf, tk, entry), lvl(ivf, tk, exit_)
                    if np.isnan(iv0) or np.isnan(iv1):
                        bad = True
                        break
                    if t["engine"] == "Variance risk premium":
                        # marked against delivered vol, not against the IV re-mark
                        rvf = lvl(rv_fwd, tk, exit_)
                        if np.isnan(rvf):
                            bad = True
                            break
                        pnl += sgn * (rvf - iv0)
                        parts.append("%s: IV %.1f at entry vs %.1f realized over the week"
                                     % (self.NAME.get(tk, tk), iv0, rvf))
                    else:
                        pnl += sgn * (iv1 - iv0)
                        parts.append("%s IV %.1f → %.1f (%+.1f)"
                                     % (self.NAME.get(tk, tk), iv0, iv1, iv1 - iv0))
                if bad:
                    d["note"] = "implied vol missing at entry or exit"
                    out.append(d); continue
                d["pnl"], d["detail"] = pnl, "; ".join(parts)

            else:  # price legs
                d["unit"] = "%"
                parts, pnl, bad = [], 0.0, False
                for tk, sgn in legs:
                    p0, p1 = lvl(close, tk, entry), lvl(close, tk, exit_)
                    if np.isnan(p0) or np.isnan(p1) or p0 == 0:
                        bad = True
                        break
                    r = (p1 / p0 - 1.0) * 100.0
                    pnl += sgn * r
                    parts.append("%s %s %.2f → %.2f (%+.2f%%)"
                                 % ("long" if sgn > 0 else "short",
                                    self.NAME.get(tk, tk), p0, p1, r))
                if bad:
                    d["note"] = "price missing at entry or exit"
                    out.append(d); continue
                d["pnl"], d["detail"] = pnl, "; ".join(parts)

            d["outcome"] = "WIN" if d["pnl"] > 1e-9 else ("LOSS" if d["pnl"] < -1e-9 else "FLAT")
            out.append(d)

        res = pd.DataFrame(out)
        return res

    # -- one-shot ----------------------------------------------------------
    def run(self, px, iv, cfg, asof, horizon=None, top_n=TOP_N,
            sectors=SECTORS, sector_n=SECTOR_N, engine_n=ENGINE_N, engines=None):
        """Replay `asof`, pick the trades, score them. -> (results, summary)."""
        sig = self.signals_asof(px, iv, cfg, asof)
        picks = self.select(sig, top_n=top_n, sectors=sectors, sector_n=sector_n,
                            engine_n=engine_n, engines=engines)
        res = self.score(picks, px, iv, cfg, asof, horizon)
        # An engine with nothing on the board is not the same as an engine that
        # lost, so carry the silent ones into the report instead of letting them
        # drop out of the breakdown.
        fired = set(sig["engine"]) if sig is not None and not sig.empty else set()
        silent = [e for e in (engines or ENGINES) if e not in fired]
        return res, summarize(res, sectors=sectors, silent_engines=silent)


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
def summarize(res, sectors=SECTORS, silent_engines=()):
    """Hit rate overall / by engine / by basis, plus confidence calibration.

    `silent_engines` are engines that produced no signal at all on the replay
    date — reported separately so a blank week is never read as a bad one.
    """
    if res is None or res.empty:
        return dict(n=0, scored=0, silent_engines=list(silent_engines))
    live = res[res["outcome"].isin(["WIN", "LOSS", "FLAT"])]
    wins = int((live["outcome"] == "WIN").sum())
    n = len(live)

    def blk(df):
        if df.empty:
            return dict(n=0, wins=0, hit=np.nan, vol_pnl=np.nan, px_pnl=np.nan)
        v = df[df["unit"] == "vol pts"]["pnl"]
        p = df[df["unit"] == "%"]["pnl"]
        return dict(n=len(df), wins=int((df["outcome"] == "WIN").sum()),
                    hit=(df["outcome"] == "WIN").mean() * 100.0,
                    vol_pnl=v.mean() if len(v) else np.nan,
                    px_pnl=p.mean() if len(p) else np.nan)

    # Keep the dashboard's engine order rather than alphabetical, and keep an
    # engine that fired but went entirely unscored visible with n=0.
    seen = [e for e in ENGINES if e in set(res["engine"])]
    seen += [e for e in sorted(set(res["engine"])) if e not in ENGINES]
    by_engine = {e: blk(live[live["engine"] == e]) for e in seen}
    by_basis = {b: blk(live[live["basis"] == b]) for b in sorted(live["basis"].unique())}
    in_sec = live[live["sectors"].apply(lambda cs: any(c in sectors for c in cs))]

    s = dict(n=len(res), scored=n, wins=wins,
             hit=(wins / n * 100.0) if n else np.nan,
             avg_conf=float(live["conf"].mean()) if n else np.nan,
             entry=res["entry_date"].iloc[0], exit=res["exit_date"].iloc[0],
             sessions=int(res["sessions"].iloc[0]),
             by_engine=by_engine, by_basis=by_basis,
             sector=blk(in_sec), overall=blk(live),
             silent_engines=list(silent_engines),
             unscored=int(len(res) - n))
    s["calibration"] = (s["hit"] - s["avg_conf"]) if n else np.nan
    return s


# ---------------------------------------------------------------------------
# Reports
# ---------------------------------------------------------------------------
def _fmt_pnl(v, unit):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "—"
    return "%+.2f %s" % (v, unit)


def report_text(res, summary, cfg=None):
    """Plain-text report — copy/pasteable, identical numbers to the HTML one."""
    if res is None or res.empty:
        return "No signals were open on the replay date, so there is nothing to score."
    L = []
    L.append("TRADE-SIGNAL OUTCOMES")
    L.append("entry %s  ->  exit %s   (%d trading sessions)"
             % (pd.Timestamp(summary["entry"]).date(),
                pd.Timestamp(summary["exit"]).date() if summary["exit"] is not None else "—",
                summary["sessions"]))
    if cfg:
        L.append("settings: %s RV %s / %dd, cheap<=%d rich>=%d, quality>=%s"
                 % (cfg.get("lb_name"), cfg.get("rv_estimator"), cfg.get("rv_window"),
                    cfg.get("iv_lo"), cfg.get("iv_hi"), cfg.get("min_quality")))
    L.append("")
    for _, r in res.iterrows():
        L.append("[%s] %s — %s" % (r["engine"], r["name"], r["side"]))
        L.append("    picked as: %s | confidence %.0f%% (n=%d) | trigger: %s"
                 % (r["basis"], r["conf"], int(r.get("conf_n") or 0), r["trigger"]))
        L.append("    outcome: %-5s  %s" % (r["outcome"], _fmt_pnl(r["pnl"], r["unit"])))
        if r["detail"]:
            L.append("    marks: %s" % r["detail"])
        if r["note"]:
            L.append("    note: %s" % r["note"])
        L.append("")
    o = summary["overall"]
    L.append("RESULTS")
    L.append("  hit rate            %s (%d of %d)"
             % ("%.0f%%" % summary["hit"] if summary["scored"] else "—",
                summary["wins"], summary["scored"]))
    L.append("  avg predicted conf  %.0f%%   -> calibration %+.0f pts"
             % (summary["avg_conf"], summary["calibration"]))
    if not np.isnan(o["vol_pnl"]):
        L.append("  avg vol-trade P&L   %+.2f vol pts" % o["vol_pnl"])
    if not np.isnan(o["px_pnl"]):
        L.append("  avg price-trade P&L %+.2f%%" % o["px_pnl"])
    L.append("")
    L.append("  by engine:")
    for e, b in summary["by_engine"].items():
        L.append("    %-24s %d/%d  %s" % (e, b["wins"], b["n"],
                 "%.0f%%" % b["hit"] if b["n"] else "—"))
    for e in summary.get("silent_engines", []):
        L.append("    %-24s no signals on this date" % e)
    L.append("  by how it was picked:")
    for b_, b in summary["by_basis"].items():
        L.append("    %-24s %d/%d  %s" % (b_, b["wins"], b["n"],
                 "%.0f%%" % b["hit"] if b["n"] else "—"))
    sec = summary["sector"]
    L.append("  metals/energy only:      %d/%d  %s"
             % (sec["wins"], sec["n"], "%.0f%%" % sec["hit"] if sec["n"] else "—"))
    if summary["unscored"]:
        L.append("  %d trade(s) could not be marked — see notes above." % summary["unscored"])
    return "\n".join(L)


def report_html(res, summary, cfg=None, theme=None):
    """Dark-themed HTML report matching the dashboard's look."""
    T = dict(THEME, **(theme or {}))
    if res is None or res.empty:
        return ("<div style='font:400 13px Inter,Arial;color:%s'>No signals were open "
                "on the replay date, so there is nothing to score.</div>" % T["MUTED"])

    def oc_color(o):
        return {"WIN": T["GREEN"], "LOSS": T["RED"], "FLAT": T["AMBER"]}.get(o, T["MUTED"])

    def conf_color(c):
        if c is None or (isinstance(c, float) and np.isnan(c)):
            return T["MUTED"]
        return T["GREEN"] if c >= 65 else (T["TEAL"] if c >= 55 else
                                           (T["AMBER"] if c >= 45 else T["RED"]))

    head = ("<div style='font:800 22px Inter,Segoe UI,Arial;color:%s'>Trade-signal outcomes</div>"
            "<div style='font:400 12px Inter,Arial;color:%s;padding:3px 0 10px'>"
            "signals as they stood on <b style='color:%s'>%s</b>, marked to "
            "<b style='color:%s'>%s</b> — %d trading sessions. Entry-close to exit-close, "
            "no costs or sizing.</div>"
            % (T["TXT"], T["MUTED"], T["TXT"], pd.Timestamp(summary["entry"]).date(),
               T["TXT"], pd.Timestamp(summary["exit"]).date() if summary["exit"] is not None else "—",
               summary["sessions"]))

    def sign_color(v):
        """Green up, red down, muted when there were no trades of that kind."""
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return T["MUTED"]
        return T["GREEN"] if v > 0 else (T["RED"] if v < 0 else T["AMBER"])

    o = summary["overall"]
    cards = [("Hit rate", "%.0f%%" % summary["hit"] if summary["scored"] else "—",
              "%d of %d marked" % (summary["wins"], summary["scored"]),
              sign_color(summary["hit"] - 50 if summary["scored"] else np.nan)),
             ("Predicted", "%.0f%%" % summary["avg_conf"], "avg confidence at entry", T["BLUE"]),
             ("Calibration", "%+.0f pts" % summary["calibration"], "realized minus predicted",
              sign_color(summary["calibration"])),
             ("Vol trades", _fmt_pnl(o["vol_pnl"], "vol pts"), "average P&L",
              sign_color(o["vol_pnl"])),
             ("Price trades", _fmt_pnl(o["px_pnl"], "%"), "average P&L",
              sign_color(o["px_pnl"]))]
    card_html = "".join(
        "<div style='background:%s;border:1px solid %s;border-radius:10px;padding:10px 14px;"
        "min-width:130px'><div style='font:700 10px Inter,Arial;color:%s;letter-spacing:.08em'>%s</div>"
        "<div style='font:800 20px Inter,Arial;color:%s;padding:2px 0'>%s</div>"
        "<div style='font:400 11px Inter,Arial;color:%s'>%s</div></div>"
        % (T["PANEL"], T["GRID"], T["MUTED"], k.upper(), c, v, T["MUTED"], sub)
        for k, v, sub, c in cards)

    rows = ""
    for _, r in res.iterrows():
        ec = ENGINE_COLOR.get(r["engine"], T["MUTED"])
        tag = {"engine coverage": "ENGINE", "metals/energy quota": "SECTOR"}.get(r["basis"])
        badge = ("<span style='font:700 9px Inter,Arial;color:%s;border:1px solid %s;"
                 "border-radius:4px;padding:1px 5px'>%s</span>"
                 % (T["AMBER"], T["AMBER"], tag)) if tag else ""
        rows += (
            "<tr style='border-top:1px solid %s'>"
            "<td style='padding:10px 12px;vertical-align:top;white-space:nowrap'>"
            "<div style='font:700 12px Inter,Arial;color:%s'>%s</div>"
            "<div style='font:400 10px Inter,Arial;color:%s;padding-top:2px'>%s %s</div></td>"
            "<td style='padding:10px 12px;vertical-align:top'>"
            "<div style='font:600 13px Inter,Arial;color:%s'>%s</div>"
            "<div style='font:400 11px Inter,Arial;color:%s;padding-top:3px'>%s</div>"
            "<div style='font:400 11px Inter,Arial;color:%s;padding-top:4px'>%s</div></td>"
            "<td style='padding:10px 12px;vertical-align:top;text-align:right;white-space:nowrap'>"
            "<div style='font:800 14px Inter,Arial;color:%s'>%.0f%%</div>"
            "<div style='font:400 10px Inter,Arial;color:%s'>n=%d</div></td>"
            "<td style='padding:10px 12px;vertical-align:top;text-align:right;white-space:nowrap'>"
            "<div style='font:800 14px Inter,Arial;color:%s'>%s</div>"
            "<div style='font:700 11px Inter,Arial;color:%s'>%s</div></td></tr>"
            % (T["GRID"], ec, r["engine"], T["MUTED"], r["basis"], badge,
               T["TXT"], "%s — %s" % (r["name"], r["side"]),
               T["MUTED"], r["trigger"],
               T["MUTED"], (r["detail"] or r["note"] or ""),
               conf_color(r["conf"]), r["conf"], T["MUTED"], int(r.get("conf_n") or 0),
               oc_color(r["outcome"]), r["outcome"],
               T["TXT"], _fmt_pnl(r["pnl"], r["unit"])))

    table = ("<table style='border-collapse:collapse;width:100%%;background:%s;"
             "border:1px solid %s;border-radius:10px;overflow:hidden;margin-top:12px'>"
             "<tr style='background:%s'>"
             "<th style='text-align:left;padding:8px 12px;font:700 10px Inter,Arial;color:%s;"
             "letter-spacing:.08em'>ENGINE</th>"
             "<th style='text-align:left;padding:8px 12px;font:700 10px Inter,Arial;color:%s;"
             "letter-spacing:.08em'>TRADE</th>"
             "<th style='text-align:right;padding:8px 12px;font:700 10px Inter,Arial;color:%s;"
             "letter-spacing:.08em'>CONF</th>"
             "<th style='text-align:right;padding:8px 12px;font:700 10px Inter,Arial;color:%s;"
             "letter-spacing:.08em'>OUTCOME</th></tr>%s</table>"
             % (T["PANEL"], T["GRID"], T["BG"], T["MUTED"], T["MUTED"], T["MUTED"],
                T["MUTED"], rows))

    def brk(title, mapping, extra=""):
        items = "".join(
            "<div style='font:400 12px Inter,Arial;color:%s;padding:3px 0'>%s "
            "<b style='color:%s'>%d/%d</b> <span style='color:%s'>%s</span></div>"
            % (T["MUTED"], k, T["TXT"], b["wins"], b["n"],
               sign_color(b["hit"] - 50 if b["n"] else np.nan),
               "%.0f%%" % b["hit"] if b["n"] else "—")
            for k, b in mapping.items())
        return ("<div style='min-width:260px'><div style='font:700 10px Inter,Arial;color:%s;"
                "letter-spacing:.08em;padding-bottom:4px'>%s</div>%s%s</div>"
                % (T["MUTED"], title.upper(), items, extra))

    sec = summary["sector"]
    silent = "".join(
        "<div style='font:400 12px Inter,Arial;color:%s;padding:3px 0'>%s "
        "<span style='font:400 11px Inter,Arial'>no signals on this date</span></div>"
        % (T["MUTED"], e) for e in summary.get("silent_engines", []))
    breakdown = ("<div style='display:flex;gap:28px;flex-wrap:wrap;margin-top:14px'>%s%s%s</div>"
                 % (brk("By engine", summary["by_engine"], silent),
                    brk("By how it was picked", summary["by_basis"]),
                    brk("Sector", {"Metals / energy": sec})))

    foot = ("<div style='font:400 11px Inter,Arial;color:%s;margin-top:14px;line-height:1.6'>"
            "Vol trades are marked in vol points, relative-value and directional trades in "
            "percent, so the two averages are not additive. Variance-risk-premium trades are "
            "marked against the volatility actually delivered over the holding week, not against "
            "the implied-vol re-mark. One week of signals is a handful of observations — read "
            "the calibration line as a sanity check, not as evidence about the engines."
            "</div>" % T["MUTED"])

    return ("<div style='background:%s;padding:16px;border-radius:12px'>%s"
            "<div style='display:flex;gap:10px;flex-wrap:wrap'>%s</div>%s%s%s</div>"
            % (T["BG"], head, card_html, table, breakdown, foot))


# ---------------------------------------------------------------------------
# Journal — so outcomes accumulate instead of being recomputed each time
# ---------------------------------------------------------------------------
JOURNAL_DIR = "trade_journal"
JOURNAL_COLS = ["entry_date", "exit_date", "sessions", "engine", "name", "side", "basis",
                "cls", "conf", "conf_n", "conf_raw", "tier", "trigger", "outcome",
                "pnl", "unit", "detail", "note", "reason"]


def save_run(res, directory=JOURNAL_DIR):
    """Append this run to the journal and return the per-run file path."""
    if res is None or res.empty:
        return None
    os.makedirs(directory, exist_ok=True)
    d = res.copy()
    d["legs"] = d["legs"].apply(lambda ls: "|".join("%s%s" % ("+" if s > 0 else "-", t)
                                                    for t, s in (ls or [])))
    d["sectors"] = d["sectors"].apply(lambda cs: "|".join(cs or []))
    cols = [c for c in JOURNAL_COLS if c in d.columns] + ["legs", "sectors"]
    d = d[cols]
    stamp = pd.Timestamp(res["entry_date"].iloc[0]).date()
    path = os.path.join(directory, "run_%s.csv" % stamp)
    d.to_csv(path, index=False)

    # Re-running the same as-of date replaces that date's rows rather than
    # appending a second copy, so the ledger stays one row per tracked trade.
    ledger = os.path.join(directory, "ledger.csv")
    if os.path.exists(ledger):
        old = pd.read_csv(ledger)
        if "entry_date" in old.columns:
            same = pd.to_datetime(old["entry_date"], errors="coerce").dt.date == stamp
            old = old[~same.fillna(False)]
        d = pd.concat([old, d], ignore_index=True, sort=False)
    d.to_csv(ledger, index=False)
    return path


def load_ledger(directory=JOURNAL_DIR):
    path = os.path.join(directory, "ledger.csv")
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()

In [ ]:
# =====================================================================
# SETUP — reuse the dashboard's engines without touching the dashboard
# =====================================================================
# Two ways this can run, both handled here:
#   * as its own notebook  -> find the dashboard .ipynb and exec its CONFIG /
#     ANALYTICS / DATA LAYER cells into NS. Its RENDERERS and CONTROLS cells are
#     skipped, so no widgets are built and no Bloomberg pull is triggered.
#   * pasted into the dashboard notebook -> the engines are already defined, so
#     NS is just this notebook's own namespace.
# Either way the signals scored below come from the dashboard's own code.
from IPython.display import HTML, display

if "build_signals" in globals():
    NS = globals()
    print("Using the engines already defined in this notebook.")
else:
    DASHBOARD = find_dashboard()
    NS = load_dashboard(DASHBOARD)
    print("Loaded %s (cells %s)." % (os.path.basename(DASHBOARD), NS["__loaded_cells__"]))

print("%d assets, %d with implied vol." % (len(NS["ALL_TICKERS"]), len(NS["IV_TICKERS"])))

In [ ]:
# =====================================================================
# DATA — one Bloomberg pull, 3y of OHLC + implied vol
# =====================================================================
# Same fetch_all the dashboard uses. Takes a couple of minutes; the result is
# cached in DATA so the cells below can be re-run freely.
px, iv, px_fail, iv_fail = NS["fetch_all"]()
DATA = dict(px=px, iv=iv)

print("prices: %d assets to %s" % (px["close"].shape[1], px["close"].index[-1].date()))
print("implied vol: %d assets" % iv.shape[1])
if px_fail:
    print("no price:", ", ".join(NS["NAME"].get(t, t) for t in px_fail))
if iv_fail:
    print("no IV:", ", ".join(NS["NAME"].get(t, t) for t in iv_fail))

In [ ]:
# =====================================================================
# SETTINGS — what to replay, and with which dashboard settings
# =====================================================================
close = DATA["px"]["close"]

HORIZON  = 5                      # trading sessions held (one week)
AS_OF    = close.index[-(HORIZON + 1)]   # replay date = one week back on the data's own calendar
TOP_N    = 5                      # highest-confidence signals to track
ENGINE_N = 1                      # ...topped up to this many per engine, so each one is judged
SECTORS  = ("Energy", "Precious Metals", "Base Metals")
SECTOR_N = 3                      # ...and to at least this many metals/energy trades

# The dashboard's default control panel. Change these to score a different
# configuration — e.g. lookback=504 for 2y percentiles, or min_conf=55 to
# replay only the ideas the panel would have shown above 55% confidence.
CFG = dict(lookback=252, lb_name="1y",
           rv_estimator="Yang-Zhang (OHLC)", rv_window=21,
           iv_lo=10, iv_hi=90, disp_z=2.0, corr_z=1.0,
           corr_window=NS["CORR_WINDOW"], pair_win=NS["PAIR_WIN"],
           ll_window=NS["LEADLAG_WINDOW"], ll_r=0.30, ll_gap=1.5,
           min_quality="Fair", exclude_stale=True, min_conf=0)

print("replaying %s, marking to %s" % (AS_OF.date(), close.index[-1].date()))

In [ ]:
# =====================================================================
# RUN — replay, pick, score
# =====================================================================
TR = Tracker(NS, horizon=HORIZON)

SIGNALS = TR.signals_asof(DATA["px"], DATA["iv"], CFG, AS_OF)
print("%d signals were on the board on %s:" % (len(SIGNALS), AS_OF.date()))
if len(SIGNALS):
    print(SIGNALS.groupby("engine").size().to_string())

RESULTS, SUMMARY = TR.run(DATA["px"], DATA["iv"], CFG, AS_OF, horizon=HORIZON,
                          top_n=TOP_N, engine_n=ENGINE_N, sectors=SECTORS, sector_n=SECTOR_N)
display(HTML(report_html(RESULTS, SUMMARY, CFG, theme=TR.theme)))

In [ ]:
# =====================================================================
# SAVE — plain-text report + CSV journal
# =====================================================================
# run_<date>.csv is this replay; ledger.csv accumulates across replays so the
# hit rate builds a sample over time. Re-running a date replaces its rows.
print(report_text(RESULTS, SUMMARY, CFG))
path = save_run(RESULTS)
print("\nsaved -> %s" % path)

In [ ]:
# =====================================================================
# OPTIONAL — walk several weeks back to build a real sample
# =====================================================================
# One week is five-ish trades: far too few to judge an engine. This replays the
# last WEEKS non-overlapping weeks and pools them. Each replay only ever sees
# data up to its own as-of date, so the pooled hit rate is still out-of-sample.
WEEKS = 8

frames = []
for k in range(1, WEEKS + 1):
    i = len(close) - 1 - k * HORIZON
    if i < 400:                                   # need history for percentiles
        break
    asof = close.index[i]
    r, s = TR.run(DATA["px"], DATA["iv"], CFG, asof, horizon=HORIZON,
                  top_n=TOP_N, engine_n=ENGINE_N, sectors=SECTORS, sector_n=SECTOR_N)
    if r is None or r.empty:
        continue
    frames.append(r)
    save_run(r)
    print("%s  %2d trades  hit %s" % (asof.date(), s["scored"],
          "%.0f%%" % s["hit"] if s["scored"] else "—"))

if frames:
    POOLED = pd.concat(frames, ignore_index=True, sort=False)
    PSUM = summarize(POOLED, sectors=SECTORS)
    print("\npooled over %d weeks: %d trades, hit rate %.0f%%, avg confidence %.0f%% "
          "(calibration %+.0f pts)"
          % (len(frames), PSUM["scored"], PSUM["hit"], PSUM["avg_conf"], PSUM["calibration"]))
    for e, b in PSUM["by_engine"].items():
        print("   %-24s %d/%d  %s" % (e, b["wins"], b["n"],
              "%.0f%%" % b["hit"] if b["n"] else "—"))

---
### How each trade is marked

All marks are entry close to exit close, no costs, no sizing, equal notional on both legs of a
pair. This measures whether the signal pointed the right way, not a tradable P&L.

| Engine | Wins when | P&L |
|---|---|---|
| IV mean-reversion | IV rises after BUY VOL, falls after SELL VOL | ±(IV_exit − IV_entry), vol pts |
| Variance risk premium | realized vol over the week comes in under the implied quoted at entry (SELL VOL) | ±(IV_entry − RV_realized), vol pts |
| Vol dispersion | the rich/cheap IV spread converges | ΔIV_cheap − ΔIV_rich, vol pts |
| Correlation RV | the laggard closes the gap on the outperformer | ret_long − ret_short, % |
| Lead-lag catch-up | the follower moves the leader's way | ±ret_follower, % |

Variance-risk-premium trades are marked against **delivered** volatility rather than the IV
re-mark, because that is what an option seller is actually paid on. It is the one engine whose
outcome is not simply "did the quote move".

### Reading the numbers

- **Calibration** is realized hit rate minus the average confidence the engines predicted at entry.
  Positive means the confidence scores were, if anything, conservative that week.
- Vol trades are in vol points and price trades in percent, so the two average-P&L figures are
  not additive and should not be summed into a portfolio return.
- Every engine that fired is guaranteed a trade, so the by-engine table is never empty for an
  engine that had something to say. An engine with **no signals at all** that day is listed
  separately as *no signals on this date* — a blank week is not a bad week.
- Trades carrying an `ENGINE` or `SECTOR` badge got in on a quota, not on merit. They drag the
  headline hit rate toward 50% by design; the *by how it was picked* table separates them from
  the top-confidence picks so you can see both numbers.
- One week is roughly five to eight trades. That is a log entry, not evidence: run the optional
  multi-week cell above before drawing any conclusion about an engine.
- Trades whose legs cannot be resolved, or whose IV is missing at either end, are reported as
  **N.A.** rather than silently dropped or marked at a guessed level.
- `trade_journal/ledger.csv` is the running record. It carries the full reason text and trigger
  for every tracked trade, so a stale entry can always be traced back to what the dashboard said.

### Where the code lives

The **TRACKER LIBRARY** cell above is the whole implementation, inlined so that this notebook and
the dashboard notebook are the only two files BQuant needs. It is generated from
`dev/tracker_source.py` in the repo by `python dev/build_notebook.py` — edit it there and
regenerate rather than editing the cell, or the next rebuild will overwrite your changes. The
`dev/` folder is developer tooling only; nothing in it has to be uploaded to BQuant.

`python dev/test_tracker.py` runs the whole replay → select → score → report path against a
generated market with no Bloomberg needed, checks the P&L signs by recomputing them
independently, asserts a replay cannot see data after its as-of date, and executes this
notebook's own cells so the inlined library cannot drift from its source.